This file contains the mathematical justifications for tensor operators implemented in code, and their gradient update policy.

**Key formula** 

$$\bold{A} \cdot (\bold{B}\bold{C}) = (\bold{A}\bold{C}^T) \cdot \bold{B} = (\bold{B}^T \bold{A}) \cdot \bold{C}$$

*proof*

We can use summation notation to explicitly write this out:

$\bold{A} \cdot (\bold{B}\bold{C}) = \sum_i\sum_k a_{ik} \sum_j b_{ij}c_{jk}$


$ = \sum_i\sum_k\sum_j a_{ik}b_{ij}c_{jk}$

Now we do some big brain thinking: what is $\bold{C}^T$? Each i,j element in the transpose is flipped across the major diagonal meaning:

$\bold{C}^T_{kj} = c_{jk}$

$ = \sum_i\sum_k\sum_j a_{ik}\bold{C}^T_{kj}b_{ij}$

$ = \sum_i\sum_j b_{ij}\sum_k (a_{ik}\bold{C}^T_{kj})$
$ = \bold{B} \cdot {\bold{A}\bold{C}^T}$

Going back to  $\sum_i\sum_k\sum_j a_{ik}b_{ij}c_{jk}$ we can do a similar thing to $\bold{B}$: $\bold{B}^T_{ji} = b_{ij}$


$\sum_i\sum_k\sum_j a_{ik}\bold{B}^T_{ji}c_{jk} = \sum_j\sum_k c_{jk}\sum_i (\bold{B}^T_{ji}a_{ik})$

$ = \bold{C} \cdot (\bold{B}^T \bold{A})$

In summary we have proven our identity:
$$\bold{A} \cdot (\bold{B}\bold{C}) = (\bold{A}\bold{C}^T) \cdot \bold{B} = (\bold{B}^T \bold{A}) \cdot \bold{C}$$

The easiest way to remember this is to look at the relative placements: $\bold{C}$ is the on right side of the matmul so it goes to the right of $\bold{A}$, and $\bold{B}$ is on the left side of matmul so its transpose goes to the left of $\bold{A}$


**Matrix Multiplication**

Consider the layer $\bold{T_{data}} = \bold{T_{input,1}}\bold{T_{input,2}}$. 

By the formula: $\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} \cdot d\bold{T_{input,2}}= \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (d\bold{T_{data}}) 
= \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (\bold{T_{input,1}}d\bold{T_{input,2}})$

By the previous formula:

$\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} \cdot (d\bold{T_{input,2}}) = \bold{T_{input,1}}^T \frac{\partial \bold{E}}{\partial \bold{T_{data}}}T \cdot d\bold{T_{input,2}}$

$\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} = \bold{T_{input,1}}^T \frac{\partial \bold{E}}{\partial \bold{T_{data}}}$

Similarly, for $\frac{\partial \bold{E}}{\partial \bold{T_{input,1}}}$ we get:
$\frac{\partial \bold{E}}{\partial \bold{T_{input,1}}} \cdot d\bold{T_{input,1}} =  \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (d\bold{T_{input,1}}\bold{T_{input,2}})$

$\implies \frac{\partial \bold{E}}{\partial \bold{T_{input,1}}} = \frac{\partial \bold{E}}{\partial \bold{T_{input,1}}}\bold{T_{input,2}}^T$



**Addition**
Consider the layer $\bold{T_{data}} = \bold{T_{input,1}} + \bold{T_{input,2}}$

Then:
$\frac{\partial E}{\partial T_{input,1}} \cdot dT_{input,1}= \frac{\partial E}{\partial T_{data}} \cdot (d\bold{T_{input,1}} + 0)$

$\implies \frac{\partial E}{\partial T_{input,1}} = \frac{\partial E}{\partial T_{data}}$

Similarly since addition is commutative: $\frac{\partial E}{\partial T_{input,2}} = \frac{\partial E}{\partial T_{data}}$

**Scalar Multiplication:**
$\bold{T_{data}} = c\bold{T_{input}}$

$\frac{\partial E}{\partial T_{input,1}} \cdot dT_{input,1}= c\frac{\partial E}{\partial T_{data}} \cdot (d\bold{T_{input,1}})$

$\frac{\partial E}{\partial T_{input,1}} = c\frac{\partial E}{\partial T_{data}}$

*what if you want the scalar to be a parameter*?

Note that c is a scalar so its dot product is just regular mult

$\frac{\partial E}{\partial c}dc= \frac{\partial E}{\partial T_{data}} \cdot (dc\bold{T_{input}}) = (\frac{\partial E}{\partial T_{data}}) \cdot (\bold{T_{input}})dc$

$\frac{\partial E}{\partial c} = (\frac{\partial E}{\partial T_{data}}) \cdot (\bold{T_{input}})$



**Activation Functions**
They take the form $\bold{T} = f(\bold{T_1})$, where $f$ is applied to each component.

I'm using a different notation for the gradient shadow (gradient notation) because its easier to type and more literal. ill change the previous partials in another update.

$\nabla_{T_1} E \cdot d\bold{T_1}= \nabla_{T} E \cdot df = \nabla_{T} E \cdot (f'(\bold{T_1}) * d{\bold{T_1}})$, where $*$ denotes element wise multiplication.

$\nabla_{T} E \cdot (f'(\bold{T_1}) * d{\bold{T_1}}) = \nabla_{T} E * f'(\bold{T_1}) \cdot d{\bold{T_1}}$

The fact above can be easily derived by looking at the element wise addition thru summation notation.

Therefore:

$$\nabla_{T_1} E = \nabla_{T} E * f'(\bold{T_1}) $$

Where $*$ denotes element wise multiplication

**Some common activation functions**
*logistic activation funcion*
$$\sigma(x) = \frac{1}{1+e^{-x}}$$

Note that $$\sigma'(x) = \sigma(x)(1-\sigma(x))$$


*relu*
$$\text{relu}(x) = \begin{cases}
x, & x > 0 \\
0, & x \le 0 \\
\end{cases}$$

*Activation*


My own custom activation function, built to be differentiable at x = 0, be positive for all x, and have a similar shape to relu

$$\text{sAct}(x) = \begin{cases}
x+1, & x > 0 \\
e^x, & x \le 0 \\
\end{cases}$$

**Normalization Function**

$$\bold{T} = \frac{\bold{T_1}}{\sum{\bold{T_1}} + \epsilon}$$

for some $\epsilon > 0$

The reason for the epsilon is to prevent division by zero. This layer is typically used in conjunction with an activation function that forces all values to be positive.

Gradient:
$$\nabla_{T_1} E = \frac{\nabla_{T} E(\sum{\bold{T_1}} + \epsilon) - \bold{1}(\nabla_{T} E \cdot \bold{T_1})}{(\sum{\bold{T_1}}+\epsilon)^2}$$

*proof*

Starting off we have

$$\nabla_{T_1} E \cdot d\bold{T_1} = \nabla_{T} E \cdot d(\frac{\bold{T_1}}{\sum{\bold{T_1}}+\epsilon})$$

Now, notice the linear operation between the scalar and the tensor. By the product rule:

$d(\frac{\bold{T_1}}{\sum{\bold{T_1}}+\epsilon}) = \frac{d\bold{T_1}}{\sum{\bold{T_1}+\epsilon}} + \bold{T_1}d(\frac{1}{\sum{\bold{T_1}}+\epsilon})$

Now we use the chain rule on the second expression. Note that a linear operator in $\R^1$ is just multiplication with a scalar

$d(\frac{1}{\sum{\bold{T_1}}+\epsilon}) = -\frac{\sum d\bold{T_1}}{(\sum{\bold{T_1}}+\epsilon)^2} = -(\bold{1} \cdot \frac{\bold{dT_1}}{(\sum{\bold{T_1}}+\epsilon)^2})$

Where $\bold{1}$ is a tensor with all 1's.

Putting that all together yeilds:
$$\nabla_{T_1} E \cdot d\bold{T_1}= \nabla_{T} E \cdot  (\frac{d\bold{T_1}}{\sum{\bold{T_1}}+\epsilon} - \frac{\bold{T_1}(\bold{1} \cdot d\bold{T_1})}{(\sum{\bold{T_1}}+\epsilon)^2})$$

Finally putting everything under the same denominator and extraction yields the expression:

$$\nabla_{T_1} E \cdot d\bold{T_1} = \frac{\nabla_{T} E(\sum{\bold{T_1}}+\epsilon) - \bold{1}(\nabla_{T} E \cdot \bold{T_1})}{(\sum{\bold{T_1}}+\epsilon)^2} \cdot d\bold{T_1}$$

Therefore we can conclude that $\nabla_{T_1} E = \frac{\nabla_{T} E(\sum{\bold{T_1}}  + \epsilon) - \bold{1}(\nabla_{T} E \cdot \bold{T_1})}{(\sum{\bold{T_1}}+\epsilon)^2}$

**Generalized dot product**
$$\bold{T} = \bold{T_1} \cdot \bold{T_2}$$

Where $\bold{T_1} \cdot \bold{T_2} = \sum_{j} {T_1}_j {T_2}_j$

$$\nabla_{T_1} E = (\nabla_{T} E)(\bold{T_2}), \nabla_{T_2} E = (\nabla_{T} E)(\bold{T_1})$$

Note that $(\nabla_{T} E)$ is a scalar and $(\bold{T_2})$ is a tensor.

*proof*

$\nabla_{T_1} E \cdot d\bold{T_1} = $


$\bold{T} = \bold{T_1}/\bold{T_2}$

$\nabla_{T_1} E \cdot d\bold{T_1} = \nabla_T E \cdot (1/\bold{T_2}) * (d\bold{T_1}) = (\nabla_T E)/\bold{T_2} \cdot \bold{T_1}$

$$\nabla_{T_1} E = (\nabla_T E)/\bold{T_2}$$

$$\nabla_{T_2} E \cdot d\bold{T_2}= \nabla_T E \cdot \bold{T_1}d(1/\bold{T_2}) = \nabla_T E \cdot (-\bold{T_1}/\bold{T_2}^2 * d\bold{T_2})$$

$$\nabla_{T_2} E = -(\nabla_T E) * \bold{T_1}/\bold{T_2}^2$$


Here * denotes element wise multiplication. 


softmax + cross entropy combo

This layer is a special combination of both softmax and cross entropy. Combining these two analytically has its benifits, as we can utlize logairthm properties to make the computation more numerically stable.
Let

$\bold{T} = -\bold{y} \cdot ln(\frac{ e^{\bold{T_1}}}{\sum_j  e^{\bold{T_1}}})$

Now note that logarithm properties hold for these tensors because its a element wise application.

$\bold{T} = -\bold{y} \cdot (ln(e^{\bold{T_1}}) - ln(\sum_j  e^{\bold{T_1}})) = - \bold{y} \cdot (\bold{T_1} - ln(\sum_j e^{\bold{T_1}}))$

This is very useful because it prevents the internal exponent step, which could result in overflow errors if the elements in $\bold{T_1}$ are big enough. However, we have to get rid of the large values in the summation as well...

To do that, we let $\bold{m} = \max_j{\bold{T_1}}$, where $\bold{m}$ is the tensor where its elements are the maximum of the old tensor when iterated through axis j.

Then, 

$ln(\sum_j e^{\bold{T_1}}) = \bold{m} + ln(\sum_j e^{\bold{T_1}-\bold{m}})$

This makes the maximum value that any element of $\bold{T_1}$ be is 0, thereby preventing NaN values associated with large exponential values.

So overall, the layer is equal to:

$$\bold{T} = - \bold{y} \cdot (\bold{T_1} - \bold{m} - ln(\sum_j e^{\bold{T_1}-\bold{m}}))$$




$\nabla_{\bold{T_1}} E \cdot d\bold{T_1}= (\nabla_{\bold{T}} E)(-\bold{y} \cdot (d\bold{T} - dln(\sum_j e^{\bold{T_1}})))$


$ = (\nabla_{\bold{T}} E)(-\bold{y} \cdot d\bold{T} - \sum_j \bold{y}_j \cdot dln(\sum_j e^{\bold{T_1}}))$

Note the sum. THis is because numpy's rules would broadcast $ dln(\sum_j e^{\bold{T_1}})$ in axis j, therefore the dot product is best represnted by this. Since $\bold{y}$ represents a distribution on axis j, the sum equates to 1.

$ = (\nabla_{\bold{T}} E)(-\bold{y} \cdot d\bold{T} - 1 \cdot dln(\sum_j e^{\bold{T_1}}))$

$ = (\nabla_{\bold{T}} E)(-\bold{y} \cdot d\bold{T} - 1 \cdot \frac{\sum_j e^{\bold{T_1}} * d\bold{T_1}}{\sum_j e^{\bold{T_1}}})$


Now we sum over the dimensions. We will let $\sum_n$ represent summing over the rest of the indices

$\sum_n (\frac{\sum_j e^{\bold{T_1}} * d\bold{T_1}}{\sum_j e^{\bold{T_1}}})_n = \sum_n\sum_j (\frac{e^{\bold{T_1}} * d\bold{T_1}}{\sum_j e^{\bold{T_1}}})_n = \sum_n\sum_j (\frac{e^{\bold{T_1}}}{\sum_j e^{\bold{T_1}}})_n * d\bold{T_1}_n $

This equal to $\frac{e^{\bold{T_1}}}{\sum_j e^{\bold{T_1}}} \cdot d\bold{T_1}$

In the end we get:


$\nabla_{\bold{T_1}} E \cdot d\bold{T_1} = (\nabla_{\bold{T}} E)(-\bold{y} \cdot d\bold{T} - \frac{e^{\bold{T_1}}}{\sum_j e^{\bold{T_1}}} \cdot d\bold{T_1})$

$$\nabla_{\bold{T_1}} = -(\bold{y}-\frac{e^{\bold{T_1}}}{\sum_j e^{\bold{T_1}}})$$

Note that the softmax layer $P$ is equal to $\frac{e^{\bold{T_1}}}{\sum_j e^{\bold{T_1}}}$, so this can be reformulated as:


$$\nabla_{\bold{T_1}} = -(\bold{y}-P) = P-\bold{y}$$

Or prediction-actual!!!